# Figures for HPML Paper
Optimization of SLMs for Efficient Code Generation

Generates all publication figures and saves them to `outputs/figures/`.

In [ ]:
import json
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

COLORS = {
    'Base SLM':          '#4C72B0',
    'Control SFT':       '#DD8452',
    'Runtime-Aware SFT': '#55A868',
}
OUT_DIR = '../outputs/figures'
os.makedirs(OUT_DIR, exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{name}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {OUT_DIR}/{name}')

In [ ]:
# Load serving benchmark data
serving = pd.read_csv('../outputs/serving_full_results.csv')

def label_model(path):
    if 'runtime_aware' in path:
        return 'Runtime-Aware SFT'
    if 'control' in path:
        return 'Control SFT'
    return 'Base SLM'

serving['model'] = serving['model_path'].apply(label_model)
serving['backend_label'] = serving['backend'].str.upper()

# Load benchmarked candidates
candidates = []
with open('../outputs/benchmarked_candidates_full.jsonl') as f:
    for line in f:
        candidates.append(json.loads(line))
candidates_df = pd.DataFrame(candidates)
passing = candidates_df[candidates_df['benchmark_passed'] == True].copy()

print(f'Serving rows: {len(serving)}')
print(f'Benchmarked candidates: {len(candidates_df)}, passing: {len(passing)}')

In [ ]:
# Ablation results from clean test split (666 problems, retrained on train/ split)
ablation = {
    'model':               ['Base SLM', 'Control SFT', 'Runtime-Aware SFT'],
    'pass_at_1':           [0.345,       0.642,          0.639],
    'median_exec_time_ms': [0.0819,      0.0807,         0.0809],  # converted to ms
    'avg_latency_s':       [0.787,       0.679,          0.687],
}

In [ ]:
# Figure 1: Pass@1 Comparison
fig, ax = plt.subplots(figsize=(7, 4))

models = ablation['model']
scores = ablation['pass_at_1']
colors = [COLORS[m] for m in models]
bars = ax.bar(models, scores, color=colors, width=0.5, edgecolor='white', linewidth=1.2)

for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 0.82)
ax.set_ylabel('Pass@1', fontsize=12)
ax.set_title('Code Correctness: Pass@1 by Model', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', labelsize=11)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

save('fig1_pass_at_1.png')

In [ ]:
# Figure 2: Generation Latency Comparison
fig, ax = plt.subplots(figsize=(7, 4))

latencies = ablation['avg_latency_s']
bars = ax.bar(models, latencies, color=colors, width=0.5, edgecolor='white', linewidth=1.2)

for bar, val in zip(bars, latencies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
            f'{val:.3f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.0)
ax.set_ylabel('Avg Generation Latency (s)', fontsize=12)
ax.set_title('Generation Latency per Problem by Model', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', labelsize=11)

save('fig2_generation_latency.png')

In [ ]:
# Figure 3: Serving Throughput — HF vs vLLM
fig, ax = plt.subplots(figsize=(9, 4))

serving_models = ['Base SLM', 'Control SFT', 'Runtime-Aware SFT']
x = np.arange(len(serving_models))
width = 0.35

hf_vals   = [serving[(serving['model'] == m) & (serving['backend'] == 'hf')]['throughput_output_tokens_per_s'].values[0] for m in serving_models]
vllm_vals = [serving[(serving['model'] == m) & (serving['backend'] == 'vllm')]['throughput_output_tokens_per_s'].values[0] for m in serving_models]

bars_hf   = ax.bar(x - width/2, hf_vals,   width, label='HuggingFace', color='#4878CF', edgecolor='white')
bars_vllm = ax.bar(x + width/2, vllm_vals, width, label='vLLM',        color='#6ACC65', edgecolor='white')

for bar, val in zip(bars_hf, hf_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars_vllm, vllm_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(serving_models, fontsize=11)
ax.set_ylabel('Throughput (tokens/s)', fontsize=12)
ax.set_title('Serving Throughput: HuggingFace vs vLLM', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

save('fig3_serving_throughput.png')

In [ ]:
# Figure 4: GPU Utilization — HF vs vLLM
fig, ax = plt.subplots(figsize=(9, 4))

hf_util   = [serving[(serving['model'] == m) & (serving['backend'] == 'hf')]['avg_gpu_util_pct'].values[0] for m in serving_models]
vllm_util = [serving[(serving['model'] == m) & (serving['backend'] == 'vllm')]['avg_gpu_util_pct'].values[0] for m in serving_models]

bars_hf   = ax.bar(x - width/2, hf_util,   width, label='HuggingFace', color='#4878CF', edgecolor='white')
bars_vllm = ax.bar(x + width/2, vllm_util, width, label='vLLM',        color='#6ACC65', edgecolor='white')

for bar, val in zip(bars_hf, hf_util):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars_vllm, vllm_util):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(serving_models, fontsize=11)
ax.set_ylim(0, 115)
ax.set_ylabel('Avg GPU Utilization (%)', fontsize=12)
ax.set_title('GPU Utilization: HuggingFace vs vLLM', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

save('fig4_gpu_utilization.png')

In [ ]:
# Figure 5: Execution Time Distribution (log scale, trimmed to data range)
fig, ax = plt.subplots(figsize=(8, 4))

times_ms = passing['median_time'].values * 1000  # convert to ms

# Trim to 99th percentile to avoid empty tail
p99 = np.percentile(times_ms, 99)
trimmed = times_ms[times_ms <= p99]
n_clipped = len(times_ms) - len(trimmed)

log_bins = np.logspace(np.log10(trimmed.min()), np.log10(trimmed.max()), 45)
ax.hist(trimmed, bins=log_bins, color='#4C72B0', edgecolor='white', alpha=0.85)

ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.3g}'))
ax.set_xlabel('Median Execution Time (ms, log scale)', fontsize=12)
ax.set_ylabel('Number of Candidates', fontsize=12)
ax.set_title(f'Distribution of Solution Execution Times\n(7,992 passing candidates, {n_clipped} outliers >p99 excluded)', fontsize=13, fontweight='bold')

ax.axvline(np.median(times_ms), color='#DD8452', linestyle='--', linewidth=1.8,
           label=f'Median: {np.median(times_ms):.3f} ms')
ax.legend(fontsize=11)

save('fig5_execution_time_distribution.png')

In [ ]:
# Figure 6: Speedup Distribution (fastest vs first-correct candidate)
by_problem = defaultdict(list)
for _, row in passing.iterrows():
    by_problem[row['dataset_index']].append(row)

speedups = []
for idx, rows in by_problem.items():
    rows_sorted_by_id   = sorted(rows, key=lambda r: r['candidate_id'])
    rows_sorted_by_time = sorted(rows, key=lambda r: r['median_time'])
    first_time   = rows_sorted_by_id[0]['median_time']
    fastest_time = rows_sorted_by_time[0]['median_time']
    if fastest_time and fastest_time > 0:
        speedups.append(first_time / fastest_time)

speedups = np.array(speedups)
print(f'Problems: {len(speedups)}')
print(f'Median speedup: {np.median(speedups):.3f}x')
print(f'Mean speedup:   {np.mean(speedups):.3f}x')
print(f'Problems >1.5x: {(speedups >= 1.5).sum()} ({(speedups >= 1.5).mean()*100:.1f}%)')

# Clip x-axis at 2.0 to zoom into where the data actually is
clip_at = 2.0
n_beyond = (speedups > clip_at).sum()
fig, ax = plt.subplots(figsize=(8, 4))

bins = np.linspace(1.0, clip_at, 40)
ax.hist(np.clip(speedups, 1.0, clip_at), bins=bins, color='#4C72B0', edgecolor='white', alpha=0.85)

ax.axvline(np.median(speedups), color='#55A868', linestyle='--', linewidth=1.8,
           label=f'Median: {np.median(speedups):.2f}x')
ax.text(0.97, 0.92, f'{n_beyond} problems >{clip_at}x\n(not shown)',
        transform=ax.transAxes, ha='right', va='top', fontsize=9, color='gray')

ax.set_xlim(1.0, clip_at)
ax.set_xlabel('Speedup (first-correct time / fastest-correct time)', fontsize=12)
ax.set_ylabel('Number of Problems', fontsize=12)
ax.set_title('Runtime-Aware Training Signal: Speedup Distribution\n(nearly all problems cluster near 1.0x)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

save('fig6_speedup_distribution.png')

In [ ]:
# Figure 9: Memory vs Throughput Tradeoff (HF vs vLLM)
fig, ax = plt.subplots(figsize=(8, 5))

backend_colors = {'hf': '#4878CF', 'vllm': '#6ACC65'}
markers = {'Base SLM': 'o', 'Control SFT': 's', 'Runtime-Aware SFT': '^'}
marker_size = 140

# Custom label offsets (points) to avoid overlap in tight clusters
# HF points are ~same memory, slightly different throughput → stagger vertically
# vLLM points are ~same memory, different throughput → stagger horizontally
label_offsets = {
    ('Base SLM',          'hf'):   (-10, -28),
    ('Runtime-Aware SFT', 'hf'):   (-10,  12),
    ('Base SLM',          'vllm'): (-90, -28),
    ('Runtime-Aware SFT', 'vllm'): (-90,  12),
    ('Control SFT',       'hf'):   (-10,  -8),
    ('Control SFT',       'vllm'): (-90,  -8),
}

for _, row in serving.iterrows():
    x_val = row['peak_cuda_memory_mb'] / 1024
    y_val = row['throughput_output_tokens_per_s']
    key = (row['model'], row['backend'])
    dx, dy = label_offsets.get(key, (8, 4))

    ax.scatter(x_val, y_val,
               color=backend_colors[row['backend']],
               marker=markers[row['model']],
               s=marker_size, zorder=5,
               edgecolors='white', linewidths=0.8)
    ax.annotate(
        row['model'],
        (x_val, y_val),
        xytext=(dx, dy),
        textcoords='offset points',
        fontsize=8.5,
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.6),
    )

# Cluster labels
hf_mem   = serving[serving['backend'] == 'hf']['peak_cuda_memory_mb'].mean() / 1024
hf_thr   = serving[serving['backend'] == 'hf']['throughput_output_tokens_per_s'].mean()
vllm_mem = serving[serving['backend'] == 'vllm']['peak_cuda_memory_mb'].mean() / 1024
vllm_thr = serving[serving['backend'] == 'vllm']['throughput_output_tokens_per_s'].mean()

ax.text(hf_mem, hf_thr + 70, 'HuggingFace\n(low memory,\nlow throughput)',
        ha='center', fontsize=9, color='#4878CF', fontweight='bold')
ax.text(vllm_mem, vllm_thr - 200, 'vLLM\n(high memory,\nhigh throughput)',
        ha='center', fontsize=9, color='#3a9a30', fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#4878CF', markersize=9, label='HuggingFace'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#6ACC65', markersize=9, label='vLLM'),
    Line2D([0], [0], marker='o', color='gray', markersize=7, linestyle='None', label='Base SLM'),
    Line2D([0], [0], marker='s', color='gray', markersize=7, linestyle='None', label='Control SFT'),
    Line2D([0], [0], marker='^', color='gray', markersize=7, linestyle='None', label='Runtime-Aware SFT'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='upper left')

ax.set_xlabel('Peak GPU Memory (GB)', fontsize=12)
ax.set_ylabel('Throughput (tokens/s)', fontsize=12)
ax.set_title('Memory–Throughput Tradeoff: HuggingFace vs vLLM', fontsize=13, fontweight='bold')

save('fig9_memory_throughput_tradeoff.png')

In [ ]:
# Figure 8: Combined Ablation — Pass@1 + Generation Latency side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: Pass@1
bars = ax1.bar(models, ablation['pass_at_1'], color=colors, width=0.5, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, ablation['pass_at_1']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_ylim(0, 0.82)
ax1.set_ylabel('Pass@1', fontsize=12)
ax1.set_title('(a) Code Correctness', fontsize=12, fontweight='bold')
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax1.tick_params(axis='x', labelsize=10)

# Right: Generation latency
bars2 = ax2.bar(models, ablation['avg_latency_s'], color=colors, width=0.5, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars2, ablation['avg_latency_s']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
             f'{val:.3f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 1.0)
ax2.set_ylabel('Avg Generation Latency (s)', fontsize=12)
ax2.set_title('(b) Generation Latency', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', labelsize=10)

fig.suptitle('Effect of Fine-Tuning on Correctness and Latency', fontsize=13, fontweight='bold', y=1.02)

save('fig8_combined_ablation.png')

In [ ]:
# Figure 7: Per-Prompt Latency — HF vs vLLM (21x speedup headline)
fig, ax = plt.subplots(figsize=(9, 4))

hf_lat   = [serving[(serving['model'] == m) & (serving['backend'] == 'hf')]['avg_latency_per_prompt_s'].values[0] for m in serving_models]
vllm_lat = [serving[(serving['model'] == m) & (serving['backend'] == 'vllm')]['avg_latency_per_prompt_s'].values[0] for m in serving_models]

bars_hf   = ax.bar(x - width/2, hf_lat,   width, label='HuggingFace', color='#4878CF', edgecolor='white')
bars_vllm = ax.bar(x + width/2, vllm_lat, width, label='vLLM',        color='#6ACC65', edgecolor='white')

for bar, val in zip(bars_hf, hf_lat):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}s', ha='center', va='bottom', fontsize=9)
for i, (bar, val) in enumerate(zip(bars_vllm, vllm_lat)):
    speedup = hf_lat[i] / val
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}s\n({speedup:.0f}x faster)', ha='center', va='bottom', fontsize=8, color='#2a7a2a')

ax.set_xticks(x)
ax.set_xticklabels(serving_models, fontsize=11)
ax.set_ylabel('Avg Latency per Prompt (s)', fontsize=12)
ax.set_title('Per-Prompt Serving Latency: HuggingFace vs vLLM', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

save('fig7_serving_latency.png')

---
## Profiling Analysis Figures
Addressing the memory-bound bottleneck, roofline analysis, and architecture implications.

In [ ]:
# Figure 10: Roofline Model — T4 Hardware Ceiling for LLM Decode Inference

BW_TBs  = 300e9 / 1e12   # 0.30 TB/s
PEAK_TF = 65.0            # TFLOPS (FP16 tensor-core)
PARAMS  = 1.5e9           # Qwen2.5-Coder-1.5B
ridge_bf16 = PEAK_TF / BW_TBs     # ≈ 217
ridge_int4 = ridge_bf16 * 4       # INT4 → 4× larger ridge

ai_range = np.logspace(-0.5, 3.3, 600)
def roofline_tf(ai, bw, peak): return np.minimum(ai * bw, peak)

flops_per_tok = 2 * PARAMS / 1e12
hf_tflops   = 233.5  * flops_per_tok   # 0.70 TFLOPS
vllm_tflops = 1782.0 * flops_per_tok   # 5.35 TFLOPS
vllm_ra_tf  = 1891.0 * flops_per_tok   # 5.67 TFLOPS

# nudge x-positions of the two vLLM points so they don't overlap
ai_hf      = 8.0
ai_vllm    = 27.0
ai_vllm_ra = 37.0

fig, ax = plt.subplots(figsize=(11, 6.5))

# rooflines
ax.loglog(ai_range, roofline_tf(ai_range, BW_TBs, PEAK_TF),
          'k-', lw=2.5, zorder=3)
ax.loglog(ai_range, roofline_tf(ai_range, BW_TBs * 4, PEAK_TF),
          'k--', lw=2.0, zorder=3)

# region shading
ax.axvspan(ai_range[0], ridge_bf16, alpha=0.07, color='#4878CF', zorder=1)
ax.axvspan(ridge_bf16, ai_range[-1], alpha=0.07, color='#6ACC65', zorder=1)

# region labels — placed in upper part of each region (y in data coords)
ax.text(3.0, 22, 'Memory-Bound\n(bandwidth-limited)',
        fontsize=11, color='#2a4e8a', fontweight='bold', alpha=0.85,
        ha='center', va='center')
ax.text(800, 22, 'Compute-Bound\n(Tensor Core-limited)',
        fontsize=11, color='#2a7a2a', fontweight='bold', alpha=0.85,
        ha='center', va='center')

# ridge markers
ax.axvline(ridge_bf16, color='gray', lw=1.2, ls=':', alpha=0.6, zorder=2)
ax.axvline(ridge_int4, color='gray', lw=1.0, ls=':', alpha=0.35, zorder=2)
ax.text(ridge_bf16 * 0.68, 0.23,
        f'Ridge (BF16)\nB≈{ridge_bf16:.0f}', fontsize=8.5,
        ha='right', va='bottom', color='gray')
ax.text(ridge_int4 * 1.08, 0.23,
        f'Ridge (INT4)\nB≈{ridge_int4:.0f}', fontsize=8.5,
        ha='left', va='bottom', color='gray', alpha=0.7)

# operating points — no label= to keep legend clean (direct annotations below)
ax.scatter(ai_hf,      hf_tflops,   color='#4878CF', s=170,
           zorder=6, edgecolors='white', linewidths=0.8)
ax.scatter(ai_vllm,    vllm_tflops, color='#6ACC65', s=170,
           zorder=6, edgecolors='white', linewidths=0.8)
ax.scatter(ai_vllm_ra, vllm_ra_tf,  color='#DD8452', s=170, marker='^',
           zorder=6, edgecolors='white', linewidths=0.8)

# direct labels on the points
ax.annotate('HF\nbatch=8', xy=(ai_hf, hf_tflops),
            xytext=(ai_hf * 2.2, hf_tflops * 0.55),
            fontsize=8.5, color='#4878CF', fontweight='bold',
            arrowprops=dict(arrowstyle='-', color='#4878CF', lw=0.8))
ax.annotate('vLLM\nBase SLM', xy=(ai_vllm, vllm_tflops),
            xytext=(ai_vllm * 0.35, vllm_tflops * 1.6),
            fontsize=8.5, color='#6ACC65', fontweight='bold', ha='right',
            arrowprops=dict(arrowstyle='-', color='#6ACC65', lw=0.8))
ax.annotate('vLLM\nRA-SFT', xy=(ai_vllm_ra, vllm_ra_tf),
            xytext=(ai_vllm_ra * 2.8, vllm_ra_tf * 0.55),
            fontsize=8.5, color='#DD8452', fontweight='bold',
            arrowprops=dict(arrowstyle='-', color='#DD8452', lw=0.8))

# legend — lines only, bottom-right
from matplotlib.lines import Line2D
leg_handles = [
    Line2D([0], [0], color='black', lw=2.5, label='T4 roofline — BF16 (observed)'),
    Line2D([0], [0], color='black', lw=2.0, ls='--', label='T4 roofline — INT4 (projected)'),
]
ax.legend(handles=leg_handles, fontsize=9, loc='lower right',
          framealpha=0.92, borderpad=0.8)

ax.set_xlabel('Arithmetic Intensity (FLOPs / byte)  ≈  Effective Batch Size', fontsize=11)
ax.set_ylabel('Attained Throughput (TFLOPS)', fontsize=11)
ax.set_title('Roofline Model: T4 GPU — LLM Decode Inference\n'
             '(Qwen2.5-Coder-1.5B, BF16 weights, 300 GB/s DRAM)',
             fontsize=13, fontweight='bold')
ax.set_xlim(ai_range[0], ai_range[-1])
ax.set_ylim(0.18, PEAK_TF * 1.4)
ax.grid(True, which='both', alpha=0.2)

save('fig10_roofline.png')

In [ ]:
# Figure 11: GPU Utilization Breakdown + Architecture Interventions
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('GPU Utilization: Bottleneck Analysis and Architecture Interventions',
             fontsize=13, fontweight='bold', y=1.01)

# ── Panel (a): activity breakdown ───────────────────────────────────────────
BW_GBs = 300.0
flops_per_tok_gb = 2 * 1.5e9 / 1e9

hf_tokps   = 233.5
vllm_tokps = 1782.0

hf_bw_pct   = hf_tokps   * flops_per_tok_gb / BW_GBs * 100
vllm_bw_pct = vllm_tokps * flops_per_tok_gb / BW_GBs * 100

obs_util = [37.6, 97.2]
labels   = ['HuggingFace\n(batch=8)', 'vLLM\n(batch=32)']

mem_pct     = [min(hf_bw_pct, obs_util[0]),   min(vllm_bw_pct, obs_util[1])]
compute_pct = [obs_util[i] - mem_pct[i] for i in range(2)]
idle_pct    = [100 - obs_util[i]        for i in range(2)]

x2 = np.arange(2)
w  = 0.45

ax1.bar(x2, idle_pct,    w, label='Idle / CPU overhead', color='#D9D9D9')
ax1.bar(x2, mem_pct,     w, bottom=idle_pct,
        label='Memory bandwidth (weights + KV-cache)', color='#4878CF', alpha=0.82)
ax1.bar(x2, compute_pct, w,
        bottom=[idle_pct[i] + mem_pct[i] for i in range(2)],
        label='Tensor-core compute', color='#6ACC65', alpha=0.82)

# HF label: inside the gray idle zone (center)
ax1.text(0, idle_pct[0] / 2,
         f'Avg GPU util\n{obs_util[0]:.1f}%',
         ha='center', va='center', fontsize=9.5, fontweight='bold', color='#333333')
# vLLM label: above the bar (idle zone is only 2.8%)
ax1.text(1, obs_util[1] + 1.5,
         f'Avg GPU util\n{obs_util[1]:.1f}%',
         ha='center', va='bottom', fontsize=9.5, fontweight='bold')

ax1.set_xticks(x2)
ax1.set_xticklabels(labels, fontsize=11)
ax1.set_ylabel('GPU Activity (%)', fontsize=11)
ax1.set_ylim(0, 118)
ax1.set_title('(a) GPU Activity Breakdown\n(decode phase, T4)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8.5, loc='lower left', framealpha=0.92, borderpad=0.7)
ax1.yaxis.grid(True, alpha=0.3)
ax1.set_axisbelow(True)
ax1.text(0.02, 0.01,
         'BW fraction from roofline; compute = observed util - BW fraction.',
         transform=ax1.transAxes, fontsize=7.5, color='gray', va='bottom')

# ── Panel (b): projected utilization ────────────────────────────────────────
interventions  = ['BF16 HF\n(baseline)', 'INT8\nweights', 'INT4\nweights',
                  'HF + larger\nbatch', 'vLLM\nBF16', 'Speculative\ndecoding', 'vLLM +\nINT4']
proj_util      = [37, 51, 65, 55, 97, 80, 99]
bar_colors     = ['#4878CF', '#5A9BD5', '#2E75B6',
                  '#9DC3E6', '#6ACC65', '#DD8452', '#375623']
edge_widths    = [2.0 if i in (0, 4) else 0.0 for i in range(len(interventions))]
edge_colors    = ['black' if e else 'white' for e in edge_widths]

bars2 = ax2.bar(np.arange(len(interventions)), proj_util,
                color=bar_colors, width=0.62, zorder=3,
                linewidth=edge_widths, edgecolor=edge_colors)

for bar, val in zip(bars2, proj_util):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 1.5,
             f'{val}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.axhline(100, color='gray', lw=1.0, ls='--', alpha=0.45)
# use axes coords to avoid right-edge clipping
ax2.text(0.02, 100 / 118 + 0.01, '100% peak',
         transform=ax2.transAxes, fontsize=8, color='gray', va='bottom', ha='left')

ax2.set_xticks(np.arange(len(interventions)))
ax2.set_xticklabels(interventions, fontsize=9)
ax2.set_ylabel('Avg GPU Utilization (%)', fontsize=11)
ax2.set_ylim(0, 118)
ax2.set_title('(b) Projected Utilization by\nArchitecture Intervention', fontsize=12, fontweight='bold')
ax2.yaxis.grid(True, alpha=0.3)
ax2.set_axisbelow(True)

obs_patch  = mpatches.Patch(facecolor='white', edgecolor='black', lw=2, label='Observed (this work)')
proj_patch = mpatches.Patch(facecolor='#9DC3E6', label='Projected')
ax2.legend(handles=[obs_patch, proj_patch], fontsize=9, loc='lower right')

plt.tight_layout()
save('fig11_decode_bottleneck.png')

In [ ]:
# Figure 12: Efficiency Cascade — Training Objective → Output Conciseness → Serving Throughput
#
# Connects the three links in the chain:
#   1. RA-SFT generates more concise code (lower HF latency = shorter outputs)
#   2. Shorter outputs → fewer decode steps per request → vLLM continuous batching
#      can cycle through more requests in the same wall-clock time
#   3. Result: RA-SFT achieves highest vLLM throughput; Control SFT (most verbose) lowest
#
# Panel (b) uses avg output tokens = vLLM throughput_tokens_per_s ÷ throughput_prompts_per_s,
# which directly measures how many tokens each model generates per completed request.
# Data: serving_full_results.csv

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle(
    'Efficiency Cascade: Training Objective  →  Output Conciseness  →  Serving Throughput',
    fontsize=13, fontweight='bold', y=1.02,
)

serving_models = ['Base SLM', 'Control SFT', 'Runtime-Aware SFT']
model_colors   = [COLORS[m] for m in serving_models]
xi = np.arange(len(serving_models))

# ── data from serving_full_results.csv ──────────────────────────────────────
hf_lat_vals = [
    serving[(serving['model'] == m) & (serving['backend'] == 'hf')]['avg_latency_per_prompt_s'].values[0]
    for m in serving_models
]
vllm_tok_s = [
    serving[(serving['model'] == m) & (serving['backend'] == 'vllm')]['throughput_output_tokens_per_s'].values[0]
    for m in serving_models
]
vllm_prompts_s = [
    serving[(serving['model'] == m) & (serving['backend'] == 'vllm')]['throughput_prompts_per_s'].values[0]
    for m in serving_models
]
# avg output tokens per request = total tokens/s ÷ prompts/s
avg_output_tokens = [t / p for t, p in zip(vllm_tok_s, vllm_prompts_s)]

vllm_throughput_vals = vllm_tok_s

# ── Panel 1: HF Generation Latency (proxy for output length) ────────────────
ax = axes[0]
bars = ax.bar(xi, hf_lat_vals, color=model_colors, width=0.55, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, hf_lat_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.004,
            f'{val:.3f}s', ha='center', va='bottom', fontsize=10.5, fontweight='bold')

# annotate RA-SFT ↔ Control SFT gap
gap = hf_lat_vals[1] - hf_lat_vals[2]
ax.annotate('', xy=(2, hf_lat_vals[2] + 0.012), xytext=(1, hf_lat_vals[1] - 0.003),
            arrowprops=dict(arrowstyle='<->', color='gray', lw=1.3,
                            connectionstyle='arc3,rad=0.22'))
ax.text(1.5, (hf_lat_vals[1] + hf_lat_vals[2]) / 2 + 0.03,
        f'−{gap/hf_lat_vals[1]*100:.1f}%\n(more concise)', ha='center', fontsize=8.5, color='gray')

ax.set_xticks(xi)
ax.set_xticklabels(serving_models, fontsize=9.5)
ax.set_ylim(0, 0.82)
ax.set_ylabel('Avg Latency / Prompt (s)', fontsize=11)
ax.set_title('(a) Generation Latency\n(HuggingFace, proxy for output length)', fontsize=12, fontweight='bold')
ax.yaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)

# ── Panel 2: Avg output tokens per request (from vLLM) ──────────────────────
ax = axes[1]
bars = ax.bar(xi, avg_output_tokens, color=model_colors, width=0.55, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, avg_output_tokens):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.6,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10.5, fontweight='bold')

# annotate that Control SFT generates most tokens
tok_gap = avg_output_tokens[1] - avg_output_tokens[2]
ax.annotate('', xy=(2, avg_output_tokens[2] + 0.8), xytext=(1, avg_output_tokens[1] - 0.5),
            arrowprops=dict(arrowstyle='<->', color='gray', lw=1.3,
                            connectionstyle='arc3,rad=0.22'))
ax.text(1.5, (avg_output_tokens[1] + avg_output_tokens[2]) / 2 + 2.5,
        f'+{tok_gap:.1f} tok\n(more verbose)', ha='center', fontsize=8.5, color='gray')

ax.set_xticks(xi)
ax.set_xticklabels(serving_models, fontsize=9.5)
ax.set_ylabel('Avg Output Tokens / Request', fontsize=11)
ax.set_title('(b) Avg Output Length\n(vLLM, tok/s ÷ prompts/s)', fontsize=12, fontweight='bold')
ax.yaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)
ax.text(0.5, 0.63,
        'More tokens per request =\nlonger decode queue per slot\n→ lower concurrency',
        ha='center', fontsize=8.5, color='#C00000', transform=ax.transAxes)

# ── Panel 3: vLLM throughput ──────────────────────────────────────────────────
ax = axes[2]
bars = ax.bar(xi, vllm_throughput_vals, color=model_colors, width=0.55, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, vllm_throughput_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 20,
            f'{val:.0f}', ha='center', va='bottom', fontsize=10.5, fontweight='bold')

# annotate RA-SFT vs Control SFT gain
gain = vllm_throughput_vals[2] - vllm_throughput_vals[1]
ax.annotate('', xy=(2, vllm_throughput_vals[2] - 30), xytext=(1, vllm_throughput_vals[1] + 30),
            arrowprops=dict(arrowstyle='->', color='#2a7a2a', lw=1.8,
                            connectionstyle='arc3,rad=-0.28'))
ax.text(1.55, (vllm_throughput_vals[1] + vllm_throughput_vals[2]) / 2 + 60,
        f'+{gain/vllm_throughput_vals[1]*100:.1f}%\nvs Control SFT',
        ha='center', fontsize=9, color='#2a7a2a', fontweight='bold')

ax.set_xticks(xi)
ax.set_xticklabels(serving_models, fontsize=9.5)
ax.set_ylabel('Throughput (tokens/s)', fontsize=11)
ax.set_ylim(0, 2200)
ax.set_title('(c) vLLM Serving Throughput\n(higher is better)', fontsize=12, fontweight='bold')
ax.yaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()
save('fig12_efficiency_cascade.png')